<div align="center">

# 📚 AI Manuscript Analyzer — Production Edition

[![Python](https://img.shields.io/badge/Python-3.10+-blue?style=for-the-badge&logo=python)](https://python.org)
[![LangChain](https://img.shields.io/badge/LangChain-0.3+-green?style=for-the-badge)](https://langchain.com)
[![ChromaDB](https://img.shields.io/badge/ChromaDB-Vector_Store-orange?style=for-the-badge)](https://trychroma.com)
[![Gradio](https://img.shields.io/badge/Gradio-5.x-yellow?style=for-the-badge)](https://gradio.app)
[![Colab](https://img.shields.io/badge/Google_Colab-FREE-red?style=for-the-badge&logo=googlecolab)](https://colab.research.google.com)

</div>

---

## 🗺️ Notebook Map

| Cell | Section | What it does |
|------|---------|-------------|
| **1** | 🔧 Dependencies | Pin-safe install — fixes all Colab conflicts |
| **2** | ⚙️ Config | All settings in one place |
| **3** | 🤖 Models | Embeddings + LLM with GPU auto-detect |
| **4** | 🗃️ RAG Engine | `ManuscriptRAG` class + ChromaDB |
| **5** | 🧠 Analysis | 8 specialized analysis modules |
| **6** | 📊 Evaluation | Retrieval quality metrics |
| **7** | 🌐 Gradio UI | Full multi-tab web interface |
| **8** | 🧪 Demo Test | Sample manuscript end-to-end test |

> **Free tier:** No API key needed. Runs entirely on Colab's free T4 GPU.

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 1 — Conflict-safe dependency installation
#
# Root cause of Colab errors:
#   • google-colab requires requests==2.32.4
#   • google-adk requires opentelemetry-sdk<1.39.0
#
# Fix: pin BOTH before installing our packages.
# ═══════════════════════════════════════════════════════════════
import subprocess, sys

def pip(*args):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *args])

print('[1/5] Pinning Colab-compatible base versions...')
pip('requests==2.32.4')
pip(
    'opentelemetry-api==1.38.0',
    'opentelemetry-sdk==1.38.0',
    'opentelemetry-exporter-otlp-proto-http==1.38.0',
    'opentelemetry-proto==1.38.0',
    'opentelemetry-exporter-otlp-proto-common==1.38.0',
)

print('[2/5] Installing LangChain stack...')
pip(
    'langchain==0.3.25',
    'langchain-community==0.3.24',
    'langchain-huggingface==0.1.2',
    'langchain-text-splitters==0.3.8',
)

print('[3/5] Installing vector store & embeddings...')
pip('chromadb==0.5.23', 'sentence-transformers==3.4.1')

print('[4/5] Installing LLM & file utilities...')
pip(
    'transformers==4.51.3',
    'accelerate==1.6.0',
    'pypdf2==3.0.1',
    'pdfplumber==0.11.4',
    'tiktoken==0.9.0',
)

print('[5/5] Installing UI & optional OpenAI backend...')
pip('gradio==5.29.1', 'openai==1.77.0', 'langchain-openai==0.3.16')

print('\n✅  All packages installed — zero dependency conflicts!')
print('   Restart runtime if prompted, then re-run from Cell 2.')


[1/5] Pinning Colab-compatible base versions...
[2/5] Installing LangChain stack...
[3/5] Installing vector store & embeddings...
[4/5] Installing LLM & file utilities...
[5/5] Installing UI & optional OpenAI backend...

✅  All packages installed — zero dependency conflicts!
   Restart runtime if prompted, then re-run from Cell 2.


In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 2 — All configuration in one place.
# ✏️  Only edit things in this cell.
# ═══════════════════════════════════════════════════════════════
import os, warnings, logging
warnings.filterwarnings('ignore')
logging.getLogger('transformers').setLevel(logging.ERROR)
logging.getLogger('chromadb').setLevel(logging.ERROR)

# ── LLM Backend ──────────────────────────────────────────────
# Leave blank for FREE HuggingFace backend.
# Paste 'sk-...' here for GPT-4o-mini (much better output).
OPENAI_API_KEY: str = ''   # <- paste key here

# ── Embedding model (always FREE) ────────────────────────────
# 'sentence-transformers/all-MiniLM-L6-v2'   22M  fast baseline
# 'BAAI/bge-small-en-v1.5'                   33M  best MTEB/size ratio
# 'sentence-transformers/all-mpnet-base-v2'  109M best quality (slower)
EMBEDDING_MODEL: str = 'sentence-transformers/all-MiniLM-L6-v2'

# ── Free LLM ─────────────────────────────────────────────────
# 'google/flan-t5-large'  750MB  fast, decent
# 'google/flan-t5-xl'     3GB    slower, better  (needs >8GB VRAM)
FREE_LLM_MODEL: str = 'google/flan-t5-large'

# ── RAG parameters ────────────────────────────────────────────
CHUNK_SIZE     : int = 800
CHUNK_OVERLAP  : int = 120
TOP_K_RETRIEVAL: int = 6

# ── Storage ───────────────────────────────────────────────────
CHROMA_DIR : str = '/content/chroma_manuscript_db'
EXPORT_DIR : str = '/content/manuscript_reports'
os.makedirs(EXPORT_DIR, exist_ok=True)

BACKEND = 'OpenAI GPT-4o-mini' if OPENAI_API_KEY else f'HuggingFace {FREE_LLM_MODEL} (FREE)'

print('╔══════════════════════════════════════════════╗')
print('║      AI Manuscript Analyzer — v2.0           ║')
print('╠══════════════════════════════════════════════╣')
print(f'║  LLM      : {BACKEND:<34}║')
print(f'║  Embeds   : {EMBEDDING_MODEL.split("/")[-1]:<34}║')
print(f'║  Chunk    : size={CHUNK_SIZE} overlap={CHUNK_OVERLAP:<18}║')
print(f'║  Top-K    : {TOP_K_RETRIEVAL:<34}║')
print('╚══════════════════════════════════════════════╝')


╔══════════════════════════════════════════════╗
║      AI Manuscript Analyzer — v2.0           ║
╠══════════════════════════════════════════════╣
║  LLM      : HuggingFace google/flan-t5-large (FREE)║
║  Embeds   : all-MiniLM-L6-v2                  ║
║  Chunk    : size=800 overlap=120               ║
║  Top-K    : 6                                 ║
╚══════════════════════════════════════════════╝


In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 3 — Load embedding model and LLM.
# Automatically uses GPU when available, falls back to CPU.
# ═══════════════════════════════════════════════════════════════
import torch
from langchain_huggingface import HuggingFaceEmbeddings

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'🖥️  Device: {DEVICE.upper()}', end='')
if DEVICE == 'cuda':
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f' ({torch.cuda.get_device_name(0)}, {vram:.1f} GB)')
else:
    print(' — enable GPU runtime for faster inference')

print(f'\n⏳ Loading embeddings [{EMBEDDING_MODEL}]...')
embeddings = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL,
    model_kwargs={'device': DEVICE},
    encode_kwargs={'normalize_embeddings': True, 'batch_size': 32},
)
print('✅ Embeddings ready')

if OPENAI_API_KEY:
    from langchain_openai import ChatOpenAI
    os.environ['OPENAI_API_KEY'] = OPENAI_API_KEY
    llm = ChatOpenAI(model='gpt-4o-mini', temperature=0.3, max_tokens=1500)
    print('✅ LLM ready: OpenAI GPT-4o-mini')
else:
    from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline as hf_pipeline
    from langchain_community.llms import HuggingFacePipeline
    print(f'\n⏳ Loading {FREE_LLM_MODEL}... (first run ~750 MB download)')
    _tok = AutoTokenizer.from_pretrained(FREE_LLM_MODEL)
    _mdl = AutoModelForSeq2SeqLM.from_pretrained(
        FREE_LLM_MODEL,
        torch_dtype=torch.float16 if DEVICE == 'cuda' else torch.float32,
        device_map='auto' if DEVICE == 'cuda' else None,
    )
    _pipe = hf_pipeline(
        'text2text-generation', model=_mdl, tokenizer=_tok,
        max_new_tokens=768, temperature=0.3, do_sample=True,
        device=0 if DEVICE == 'cuda' else -1,
    )
    llm = HuggingFacePipeline(pipeline=_pipe)
    print(f'✅ LLM ready: {FREE_LLM_MODEL}')

print('\n🎉 All models loaded — proceed to Cell 4')


🖥️  Device: CUDA (Tesla T4, 15.6 GB)

⏳ Loading embeddings [sentence-transformers/all-MiniLM-L6-v2]...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

special_tokens_map.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.13G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

ValueError: The model has been loaded with `accelerate` and therefore cannot be moved to a specific device. Please discard the `device` argument when creating your pipeline object.

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 4 — ManuscriptRAG: production-grade RAG pipeline
#
# Pipeline:
#   Input (PDF / TXT)
#     -> _clean_text()  (normalize whitespace, strip control chars)
#     -> RecursiveCharacterTextSplitter  (800-token semantic chunks)
#     -> HuggingFace Embeddings  (384-dim cosine vectors)
#     -> ChromaDB  (HNSW index, cosine metric, persisted to disk)
#     -> MMR Retriever  (diversity-weighted top-K)
#     -> RetrievalQA  (task-specific prompt templates)
#     -> Structured text output
#
# Key design choices explained in comments below.
# ═══════════════════════════════════════════════════════════════
import os, re, uuid, shutil, time
from dataclasses import dataclass
from typing import Optional

import pdfplumber, PyPDF2
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.schema import Document
from langchain.chains import RetrievalQA
from langchain.prompts import PromptTemplate
from langchain_community.vectorstores import Chroma


@dataclass
class IngestionStats:
    source: str
    total_chars: int
    total_chunks: int
    est_pages: int
    elapsed_sec: float

    def __str__(self):
        return (
            f'  Source     : {self.source}\n'
            f'  Characters : {self.total_chars:,}\n'
            f'  Chunks     : {self.total_chunks} '
            f'(size={CHUNK_SIZE}, overlap={CHUNK_OVERLAP})\n'
            f'  Est. pages : ~{self.est_pages}\n'
            f'  Time       : {self.elapsed_sec:.2f}s'
        )


class ManuscriptRAG:
    """
    Production RAG engine for literary manuscript analysis.

    Design decisions:
    - RecursiveCharacterTextSplitter tries paragraph->sentence->word
      boundaries in order, preserving semantic coherence per chunk.
    - MMR (Maximal Marginal Relevance) retrieval returns diverse chunks,
      not just the top-6 most similar (which would often be near-duplicates).
    - Task-specific prompt templates per analysis module improve output
      quality significantly vs. a single generic prompt.
    - pdfplumber is preferred over PyPDF2 for better table/column layout
      handling; PyPDF2 is the fallback for encrypted/complex PDFs.
    """

    SYSTEM_PREFIX = (
        'You are a senior literary editor and publishing consultant with '
        '20 years of experience. Provide authoritative, specific, and '
        'actionable analysis based solely on the manuscript excerpts below.\n\n'
    )

    def __init__(self):
        self._splitter = RecursiveCharacterTextSplitter(
            chunk_size=CHUNK_SIZE,
            chunk_overlap=CHUNK_OVERLAP,
            separators=['\n\n\n', '\n\n', '\n', '. ', '! ', '? ', ' ', ''],
            length_function=len,
        )
        self._vectorstore : Optional[Chroma] = None
        self._retriever   = None
        self.stats        : Optional[IngestionStats] = None

    # ── Ingestion ─────────────────────────────────────────────
    def ingest_text(self, text: str, source: str = 'manuscript') -> IngestionStats:
        t0   = time.time()
        text = self._clean_text(text)
        docs = self._splitter.create_documents(
            texts=[text],
            metadatas=[{'source': source, 'chunk_id': str(uuid.uuid4()),
                        'ingested_at': time.strftime('%Y-%m-%d %H:%M:%S')}]
        )
        for i, doc in enumerate(docs):
            doc.metadata['chunk_index'] = i
        self._add_to_store(docs)
        self.stats = IngestionStats(
            source=source, total_chars=len(text),
            total_chunks=len(docs),
            est_pages=max(1, len(text) // 2000),
            elapsed_sec=time.time() - t0,
        )
        return self.stats

    def ingest_pdf(self, pdf_path: str) -> IngestionStats:
        text = self._extract_pdf(pdf_path)
        return self.ingest_text(text, source=os.path.basename(pdf_path))

    # ── Query ─────────────────────────────────────────────────
    def query(self, question: str, task_prompt: str = '') -> str:
        if self._retriever is None:
            return '⚠️  No manuscript loaded. Please upload one first.'
        body = task_prompt or (
            'Context from manuscript:\n{context}\n\n'
            'Question: {question}\n\nDetailed answer:'
        )
        prompt = PromptTemplate(
            input_variables=['context', 'question'],
            template=self.SYSTEM_PREFIX + body,
        )
        chain = RetrievalQA.from_chain_type(
            llm=llm, chain_type='stuff',
            retriever=self._retriever,
            chain_type_kwargs={'prompt': prompt},
            return_source_documents=False,
        )
        result = chain.invoke({'query': question})
        return result.get('result', str(result)).strip()

    def get_similar_chunks(self, query: str, k: int = 3) -> list:
        if self._vectorstore is None:
            return []
        return [d.page_content for d in
                self._vectorstore.similarity_search(query, k=k)]

    def reset(self) -> None:
        self._vectorstore = None
        self._retriever   = None
        self.stats        = None
        if os.path.exists(CHROMA_DIR):
            shutil.rmtree(CHROMA_DIR)

    # ── Private helpers ───────────────────────────────────────
    @staticmethod
    def _clean_text(text: str) -> str:
        text = re.sub(r'\r\n', '\n', text)
        text = re.sub(r'[ \t]+', ' ', text)
        text = re.sub(r'\n{4,}', '\n\n\n', text)
        text = re.sub(r'[\x00-\x08\x0b-\x1f\x7f]', '', text)
        return text.strip()

    @staticmethod
    def _extract_pdf(path: str) -> str:
        text = ''
        try:
            with pdfplumber.open(path) as pdf:
                for i, page in enumerate(pdf.pages):
                    text += f'\n[Page {i+1}]\n{page.extract_text() or ""}'
        except Exception:
            with open(path, 'rb') as f:
                reader = PyPDF2.PdfReader(f)
                for i, page in enumerate(reader.pages):
                    text += f'\n[Page {i+1}]\n{page.extract_text() or ""}'
        return text

    def _add_to_store(self, docs: list) -> None:
        if self._vectorstore is None:
            self._vectorstore = Chroma.from_documents(
                documents=docs, embedding=embeddings,
                persist_directory=CHROMA_DIR,
                collection_metadata={'hnsw:space': 'cosine'},
            )
        else:
            self._vectorstore.add_documents(docs)
        # MMR balances relevance + diversity in retrieved chunks
        self._retriever = self._vectorstore.as_retriever(
            search_type='mmr',
            search_kwargs={'k': TOP_K_RETRIEVAL,
                           'fetch_k': TOP_K_RETRIEVAL * 3,
                           'lambda_mult': 0.7},
        )


rag = ManuscriptRAG()
print('✅ ManuscriptRAG engine initialized')
print('   Retrieval : MMR (Maximal Marginal Relevance)')
print('   Metric    : cosine similarity')
print('   Chunking  : RecursiveCharacterTextSplitter')


In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 5 — Eight specialized analysis modules.
#
# Each uses a task-specific prompt template engineered for that
# analysis type. This is a key RAG optimization: task-specific
# prompts outperform generic prompts significantly.
# ═══════════════════════════════════════════════════════════════

def generate_summary() -> str:
    return rag.query(
        question='Write a comprehensive editorial summary.',
        task_prompt=(
            'Manuscript context:\n{context}\n\n'
            'Write a 3-paragraph editorial summary for a publisher catalog.\n'
            'Para 1: Core premise and narrative hook.\n'
            'Para 2: Narrative arc — setup, conflict, stakes.\n'
            'Para 3: What makes this work distinctive.\n\n'
            'Question: {question}\nSummary:'
        )
    )

def classify_genre() -> str:
    return rag.query(
        question='Classify the genre and sub-genres of this manuscript.',
        task_prompt=(
            'Manuscript context:\n{context}\n\n'
            'Provide a structured genre classification:\n'
            'PRIMARY GENRE: [genre]\n'
            'SUB-GENRES: [up to 3]\n'
            'SHELF PLACEMENT: [bookstore section]\n'
            'EVIDENCE: [2-3 textual examples]\n'
            'CONFIDENCE: [High/Medium/Low] — [brief reason]\n\n'
            'Question: {question}\nClassification:'
        )
    )

def extract_themes() -> str:
    return rag.query(
        question='Identify and analyze the major themes.',
        task_prompt=(
            'Manuscript context:\n{context}\n\n'
            'Identify the top 5 themes. For each:\n'
            'THEME [N]: [Name]\n'
            '  Manifestation: How it appears in the narrative\n'
            '  Evidence: Specific passage or scene\n'
            '  Significance: Why it matters to the work\n\n'
            'Question: {question}\nTheme Analysis:'
        )
    )

def analyze_characters() -> str:
    return rag.query(
        question='Analyze all significant characters in this manuscript.',
        task_prompt=(
            'Manuscript context:\n{context}\n\n'
            'For each major and significant minor character:\n'
            'NAME: | ROLE: Protagonist/Antagonist/Supporting\n'
            'TRAITS: [3-5 defining traits]\n'
            'ARC: [How they change or resist change]\n'
            'FUNCTION: [Narrative purpose]\n'
            'RELATIONSHIPS: [Key connections]\n\n'
            'Question: {question}\nCharacter Analysis:'
        )
    )

def generate_marketing() -> str:
    return rag.query(
        question="Generate a complete publisher's marketing package.",
        task_prompt=(
            'Manuscript context:\n{context}\n\n'
            'Create a full marketing package:\n'
            'ELEVATOR PITCH (25 words max): One sentence sell.\n'
            'BACK COVER BLURB (100 words): With hook.\n'
            'TARGET AUDIENCE: Primary and secondary demographics.\n'
            'COMP TITLES: 3 titles with "Fans of X will love..." framing.\n'
            'KEY SELLING POINTS: 5 bullets for the sales team.\n'
            'SOCIAL MEDIA: 2 tweet-length hooks + 5 hashtags.\n'
            'AWARD POTENTIAL: Eligible categories.\n\n'
            'Question: {question}\nMarketing Package:'
        )
    )

def analyze_style() -> str:
    return rag.query(
        question='Analyze the writing style, voice, and craft.',
        task_prompt=(
            'Manuscript context:\n{context}\n\n'
            'Craft-level editorial assessment:\n'
            'NARRATIVE VOICE: POV, distance, reliability\n'
            'PROSE STYLE: Sentence rhythm, complexity, distinctive features\n'
            'DIALOGUE: Quality, character distinctiveness, pacing\n'
            'PACING: Tension building and release; scene vs summary balance\n'
            'WORLD-BUILDING: How setting is established\n'
            'STRENGTHS: Top 3 craft strengths with evidence\n'
            'REVISION NOTES: Top 3 editorial suggestions\n\n'
            'Question: {question}\nStyle Analysis:'
        )
    )

def analyze_structure() -> str:
    return rag.query(
        question='Analyze the narrative structure and plot architecture.',
        task_prompt=(
            'Manuscript context:\n{context}\n\n'
            'Map the narrative structure:\n'
            'STORY FRAMEWORK: Best-fit structure (3-act, Hero Journey, etc.)\n'
            'OPENING HOOK: Effectiveness of first pages\n'
            'INCITING INCIDENT: What + when it occurs\n'
            'MIDPOINT SHIFT: Central turning point or revelation\n'
            'CLIMAX: Highest tension point and resolution\n'
            'SUBPLOTS: Secondary storylines and weave with main plot\n'
            'STRUCTURAL ISSUES: Pacing problems, unresolved threads\n\n'
            'Question: {question}\nStructural Analysis:'
        )
    )

def sensitivity_review() -> str:
    return rag.query(
        question='Provide an editorial sensitivity and content review.',
        task_prompt=(
            'Manuscript context:\n{context}\n\n'
            'Editorial sensitivity review:\n'
            'CONTENT WARNINGS: Themes requiring advisories\n'
            'REPRESENTATION: How marginalized groups are portrayed\n'
            'CULTURAL ACCURACY: Elements needing expert review\n'
            'SENSITIVITY READERS: Recommended specialist readers\n'
            'AGE RATING: Appropriate reader age range with reasoning\n\n'
            'Question: {question}\nSensitivity Review:'
        )
    )

def custom_query(question: str) -> str:
    if not question or not question.strip():
        return 'Please enter a question about the manuscript.'
    return rag.query(question)


ANALYSIS_REGISTRY = {
    '📝 Executive Summary'    : generate_summary,
    '🏷️ Genre Classification' : classify_genre,
    '🎭 Theme Extraction'     : extract_themes,
    '🧑 Character Analysis'   : analyze_characters,
    '📣 Marketing Package'    : generate_marketing,
    '✍️ Writing Style & Craft': analyze_style,
    '🏗️ Narrative Structure'  : analyze_structure,
    '🔍 Sensitivity Review'   : sensitivity_review,
}

print(f'✅ {len(ANALYSIS_REGISTRY)} analysis modules ready:')
for k in ANALYSIS_REGISTRY:
    print(f'   {k}')


In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 6 — RAG Quality Evaluation
#
# Measures:
#   1. Ingestion stats (chunks, chars, estimated pages)
#   2. Retrieval relevance (cosine similarity distribution)
#   3. MMR diversity score (how non-redundant the chunks are)
#
# Run evaluate_rag_pipeline() after ingesting a manuscript.
# ═══════════════════════════════════════════════════════════════
import numpy as np

def evaluate_rag_pipeline(test_queries: list = None) -> dict:
    if rag._vectorstore is None:
        print('⚠️  No manuscript loaded.')
        return {}

    metrics = {}

    if rag.stats:
        s = rag.stats
        metrics['ingestion'] = {
            'source': s.source, 'total_chars': s.total_chars,
            'total_chunks': s.total_chunks, 'est_pages': s.est_pages,
            'chars_per_chunk': round(s.total_chars / max(s.total_chunks, 1)),
            'ingestion_sec': round(s.elapsed_sec, 2),
        }

    if test_queries is None:
        test_queries = [
            'Who are the main characters?',
            'What is the central conflict?',
            'Describe the setting and atmosphere.',
            'What are the major themes?',
        ]

    relevance_scores, diversity_scores = [], []
    for q in test_queries:
        results = rag._vectorstore.similarity_search_with_score(q, k=TOP_K_RETRIEVAL)
        if not results:
            continue
        scores = [1 - s for _, s in results]
        relevance_scores.extend(scores)
        texts  = [doc.page_content for doc, _ in results]
        vecs   = np.array(embeddings.embed_documents(texts))
        norms  = np.linalg.norm(vecs, axis=1, keepdims=True)
        normed = vecs / (norms + 1e-9)
        sim    = normed @ normed.T
        n = len(vecs)
        if n > 1:
            off = (sim.sum() - np.trace(sim)) / (n * (n - 1))
            diversity_scores.append(float(1 - off))

    if relevance_scores:
        metrics['retrieval'] = {
            'avg_relevance': round(float(np.mean(relevance_scores)), 4),
            'min_relevance': round(float(np.min(relevance_scores)),  4),
            'max_relevance': round(float(np.max(relevance_scores)),  4),
            'mmr_diversity': round(float(np.mean(diversity_scores)) if diversity_scores else 0, 4),
            'queries_tested': len(test_queries),
        }

    print('═' * 52)
    print('  RAG PIPELINE EVALUATION')
    print('═' * 52)
    if 'ingestion' in metrics:
        i = metrics['ingestion']
        print(f'  Source        : {i["source"]}')
        print(f'  Characters    : {i["total_chars"]:,}')
        print(f'  Chunks stored : {i["total_chunks"]}')
        print(f'  Est. pages    : ~{i["est_pages"]}')
        print(f'  Chars/chunk   : {i["chars_per_chunk"]}')
        print(f'  Ingestion     : {i["ingestion_sec"]}s')
    if 'retrieval' in metrics:
        r = metrics['retrieval']
        print(f'  Avg relevance : {r["avg_relevance"]:.4f}  (max 1.0)')
        print(f'  Score range   : {r["min_relevance"]:.4f} – {r["max_relevance"]:.4f}')
        print(f'  MMR diversity : {r["mmr_diversity"]:.4f}  (higher = less redundant)')
        avg = r['avg_relevance']
        print(f'  Rating        : {"🟢 Excellent" if avg > 0.6 else "🟡 Good" if avg > 0.4 else "🔴 Needs tuning"}')
    print('═' * 52)
    return metrics

print('✅ Evaluation module ready')
print('   Call evaluate_rag_pipeline() after ingesting a manuscript.')


In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 7 — Gradio Web Interface
# Run this cell to launch. You get a public *.gradio.live link.
# ═══════════════════════════════════════════════════════════════
import gradio as gr
import json, datetime

def handle_pdf(f):
    if f is None:
        return gr.update(value='⚠️  No file selected.', visible=True)
    try:
        s = rag.ingest_pdf(f.name)
        return gr.update(value=f'✅  PDF ingested!\n\n{s}', visible=True)
    except Exception as e:
        return gr.update(value=f'❌  {e}', visible=True)

def handle_text(t):
    if not t or not t.strip():
        return gr.update(value='⚠️  No text provided.', visible=True)
    try:
        s = rag.ingest_text(t, source='pasted_manuscript')
        return gr.update(value=f'✅  Text ingested!\n\n{s}', visible=True)
    except Exception as e:
        return gr.update(value=f'❌  {e}', visible=True)

def handle_reset():
    rag.reset()
    return '🗑️  Database cleared. Ready for a new manuscript.'

def handle_analysis(task):
    fn = ANALYSIS_REGISTRY.get(task)
    return fn() if fn else '⚠️  Unknown task.'

def handle_export(text, task):
    if not text.strip():
        return 'Nothing to export.'
    ts   = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
    slug = task.replace(' ', '_')[:30]
    path = os.path.join(EXPORT_DIR, f'report_{slug}_{ts}.txt')
    with open(path, 'w', encoding='utf-8') as f:
        f.write(f'AI Manuscript Analyzer — {task}\n')
        f.write(f'Generated: {datetime.datetime.now().isoformat()}\n')
        f.write('=' * 60 + '\n\n')
        f.write(text)
    return path

def handle_eval():
    if rag._vectorstore is None:
        return 'Ingest a manuscript first.'
    return json.dumps(evaluate_rag_pipeline(), indent=2)

def handle_chunks(q):
    if not q.strip():
        return 'Enter a search query.'
    chunks = rag.get_similar_chunks(q, k=3)
    if not chunks:
        return 'No manuscript loaded.'
    return '\n\n'.join([f'--- Chunk {i+1} ---\n{c}' for i, c in enumerate(chunks)])

TASKS = list(ANALYSIS_REGISTRY.keys())

with gr.Blocks(
    theme=gr.themes.Soft(
        primary_hue='slate', secondary_hue='blue',
        font=[gr.themes.GoogleFont('DM Sans'), 'sans-serif'],
    ),
    title='📚 AI Manuscript Analyzer',
    analytics_enabled=False,
) as demo:

    gr.Markdown(
        '# 📚 AI Manuscript Analyzer  `v2.0`\n'
        'Publishing-grade AI — **LangChain · ChromaDB · HuggingFace RAG**'
    )

    with gr.Tab('📤 Step 1 — Load Manuscript'):
        gr.Markdown('### Load your manuscript into the ChromaDB vector database')
        with gr.Row():
            with gr.Column():
                gr.Markdown('#### 📄 Option A — Upload PDF')
                pdf_file   = gr.File(label='Select PDF', file_types=['.pdf'])
                pdf_btn    = gr.Button('Ingest PDF →', variant='primary', size='lg')
                pdf_status = gr.Textbox(label='Status', lines=6, interactive=False, visible=False)
            with gr.Column():
                gr.Markdown('#### 📋 Option B — Paste Raw Text')
                raw_text    = gr.Textbox(label='Paste text', placeholder='Paste manuscript here...', lines=10)
                text_btn    = gr.Button('Ingest Text →', variant='primary', size='lg')
                text_status = gr.Textbox(label='Status', lines=6, interactive=False, visible=False)
        with gr.Row():
            reset_btn    = gr.Button('🗑️ Reset Database', variant='stop')
            reset_status = gr.Textbox(label='', interactive=False, scale=3)
        pdf_btn.click(handle_pdf,   inputs=pdf_file,  outputs=pdf_status)
        text_btn.click(handle_text, inputs=raw_text,   outputs=text_status)
        reset_btn.click(handle_reset, outputs=reset_status)

    with gr.Tab('🔍 Step 2 — Analyze'):
        gr.Markdown('### Run AI-powered editorial analysis on your manuscript')
        choice  = gr.Radio(choices=TASKS, label='Select Analysis Module', value=TASKS[0])
        run_btn = gr.Button('▶  Run Analysis', variant='primary', size='lg')
        output  = gr.Textbox(label='Result', lines=22, show_copy_button=True)
        with gr.Row():
            exp_btn  = gr.Button('💾 Export to File')
            exp_path = gr.Textbox(label='Saved to', interactive=False, scale=3)
        run_btn.click(handle_analysis, inputs=choice,           outputs=output)
        exp_btn.click(handle_export,   inputs=[output, choice], outputs=exp_path)

    with gr.Tab('💬 Custom Q&A'):
        gr.Markdown('### Ask anything about the manuscript')
        q_in  = gr.Textbox(label='Question', placeholder='Type any question...', lines=2)
        q_btn = gr.Button('Ask →', variant='primary')
        q_out = gr.Textbox(label='Answer', lines=14, show_copy_button=True)
        q_btn.click(custom_query, inputs=q_in, outputs=q_out)
        q_in.submit(custom_query, inputs=q_in, outputs=q_out)

    with gr.Tab('📊 Diagnostics'):
        gr.Markdown('### RAG Pipeline Quality Metrics')
        with gr.Row():
            with gr.Column():
                eval_btn = gr.Button('Run Evaluation', variant='secondary')
                eval_out = gr.Code(label='Metrics JSON', language='json', lines=20)
                eval_btn.click(handle_eval, outputs=eval_out)
            with gr.Column():
                chunk_q   = gr.Textbox(label='Semantic search query', placeholder='characters, conflict...')
                chunk_btn = gr.Button('Find Similar Chunks')
                chunk_out = gr.Textbox(label='Top 3 Chunks', lines=20, show_copy_button=True)
                chunk_btn.click(handle_chunks, inputs=chunk_q, outputs=chunk_out)

    with gr.Tab('📐 Architecture'):
        gr.Markdown("""
## System Architecture
```
PDF/TXT Input
    ↓
pdfplumber / PyPDF2 text extraction
    ↓
_clean_text()  — normalize whitespace, strip control chars
    ↓
RecursiveCharacterTextSplitter  (chunk=800, overlap=120)
    ↓
HuggingFace Embeddings  (all-MiniLM-L6-v2, 384-dim, cosine)
    ↓
ChromaDB  (HNSW index, persisted to /content/chroma_manuscript_db)
    ↓
MMR Retriever  (k=6, fetch_k=18, lambda=0.7)
    ↓
RetrievalQA  +  task-specific PromptTemplate
    ↓
LLM  (flan-t5-large FREE  |  GPT-4o-mini PAID)
    ↓
Structured editorial output
```
## Key Technical Decisions
| Component | Choice | Reason |
|-----------|--------|--------|
| Splitter | `RecursiveCharacterTextSplitter` | Paragraph→sentence→word fallback preserves semantics |
| Embeddings | `all-MiniLM-L6-v2` | 5× faster than BERT, near-BERT quality, 384-dim |
| Vector DB | ChromaDB | Local, zero auth, HNSW index, cosine metric |
| Retrieval | MMR | Reduces redundancy vs pure cosine top-K |
| PDF extract | pdfplumber → PyPDF2 | pdfplumber handles tables/columns; PyPDF2 is fallback |
| Prompting | Task-specific templates | Higher output quality per analysis type |
""")

demo.launch(share=True, debug=False, show_error=True, quiet=True)
print('\n🌐 UI live — use the public gradio.live link above!')


In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 8 — End-to-end demo with built-in sample manuscript.
# Run this without uploading anything to verify the full pipeline.
# ═══════════════════════════════════════════════════════════════
import textwrap

SAMPLE = """
THE CARTOGRAPHER OF UNMADE PLACES

Chapter 1

The last map Sera Voss ever drew was of a country that did not exist yet.
She worked by lamplight, the way her father had taught her, with the same steady
hand she had used to chart coastlines for the Meridian Institute for eighteen years.
But this map had no shoreline verified by ship, no mountain range measured by
theodolite. This map was made of grief, and grief, she was discovering, had its
own precise geography.

Her colleague Dayo Adeyemi appeared in her doorway holding two cups of terrible
coffee. "You're mapping it again," he said. Not a question.
"I'm working," Sera said.
"Those are different things." He set one cup beside her. "That's the village.
The one from your mother's stories."

Chapter 2

Sera's mother Ona had died in the spring, leaving behind forty years of
correspondence and a recurring story about a village in the Balkan interior
where women kept maps inside their bodies, encoded in the iron content of their blood.

Preposterous. The kind of magical thinking grief produces when it needs somewhere
to put itself. But in Ona's suitcase Sera had found a journal, and in the journal
coordinates, and the coordinates pointed to a valley her thirty years of
professional cartography had somehow never charted.

Chapter 3

Director Halverson took unusual interest when Sera presented her research.
A compact, colorless man who had never been curious about anything that did not
affect the quarterly review.
"A village that appears in no cartographic record," he said. "That is either
a remarkable oversight or a remarkable secret."
"I think it is both," Sera said.

The themes of memory versus documentation, the politics of what gets recorded
and what gets erased, ran beneath every conversation at the Institute. Sera had
spent her career making the invisible visible. She had not expected her mother's
death to become a map problem. But grief, she was learning, always finds the
form you already know how to hold.

Chapter 4

Dayo argued against the expedition with the thoroughness of someone who already
knew he had lost. He cited budget constraints, political instability, and
Sera's state of mind, carefully, in the manner of someone defusing something.
"I'm not fragile," she told him.
"I know. That's what worries me. Fragile things stay home. The ones who go are
the ones who have already decided the risk is acceptable."

She left on a Thursday. The valley, per her mother's coordinates, lay three
days by road and a day's walk along a path described as easy to find and
impossible to follow without knowing what you were looking for.
"""

print('Resetting and ingesting sample manuscript...')
rag.reset()
stats = rag.ingest_text(SAMPLE, source='cartographer_sample.txt')
print(stats)
print()
evaluate_rag_pipeline()

print('\n' + '═'*60)
print('  RUNNING ALL 8 ANALYSIS MODULES')
print('═'*60)

for label, fn in ANALYSIS_REGISTRY.items():
    print(f'\n{"─"*60}')
    print(f'  {label}')
    print(f'{"─"*60}')
    result = fn()
    for line in result.splitlines():
        print(textwrap.fill(line, width=70, subsequent_indent='    '))

print('\n\n✅ All 8 modules working!')
print('   Upload your own manuscript in Cell 7 (Gradio UI) for real analysis.')
